# A Covariance Matrix With a Negative Eigenvalue

**Topic:** Symmetric eigendecomposition &nbsp;·&nbsp; **Difficulty:** Medium

A covariance matrix arrives from a legacy pipeline. Mathematically it must be
positive semidefinite, but rounding in the pipeline has left a few eigenvalues
slightly negative. Downstream code takes square roots and produces `NaN`.

Your job is to repair it: find the nearest positive semidefinite matrix by
clipping the offending eigenvalues to zero, then measure how much accuracy is
lost when you keep only the 10 largest eigenvalues.

## The data (created in the next cell)

- `C` `(60, 60)`: symmetric, but with a handful of small negative eigenvalues.

## Your task

With **NumPy only**:

1. Use `np.linalg.eigh` (**not** `eig`, since the matrix is symmetric, and `eigh`
   returns real, sorted eigenvalues) to decompose `C`.
2. Clip every negative eigenvalue to `0` and rebuild the matrix from the clipped
   spectrum.
3. Also build the rank-10 truncation, keeping only the 10 largest eigenvalues.
4. Set `answers` to a dict with `"n_negative"` (int, eigenvalues `< 0`),
   `"C_psd"` `(60, 60)`, `"rank10"` `(60, 60)`, and `"rel_error_rank10"`, the
   Frobenius norm of `C_psd - rank10` divided by that of `C_psd`.

## Evaluation

Matrices are compared to 1e-6. The evaluation cell also checks `C_psd` has no
negative eigenvalues.

In [ ]:
import numpy as np

rng = np.random.default_rng(67)
d = 60
F = rng.normal(0, 1, (d, 12))
C = F @ F.T + np.diag(rng.uniform(0.05, 0.4, d))
w, V = np.linalg.eigh(C)
w[:5] = [-0.03, -0.02, -0.011, -0.004, -0.001]     # the pipeline's rounding
C = V @ np.diag(w) @ V.T
C = (C + C.T) / 2
print(f"C {C.shape}, min eigenvalue = {np.linalg.eigvalsh(C).min():.4f}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the matrix `C` created above, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("n_negative", [], 5.0, -0.8371900740724958, 1e-06),
    ("C_psd", [60, 60], 506.13640960837796, 26.121319098929106, 1e-06),
    ("rank10", [60, 60], 445.81209158091275, 74.56934061216957, 1e-06),
    ("rel_error_rank10", [], 0.14513556656200513, -0.024301211144119744, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "C_psd" in answers:
    _w = np.linalg.eigvalsh(np.asarray(answers["C_psd"], dtype=float))
    print(f"  {'C_psd is PSD':<24} {'ok' if _w.min() > -1e-8 else 'wrong'}  (min eig = {_w.min():.2e})")
    _ok = _ok and _w.min() > -1e-8

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

For a symmetric matrix `eigh` is both faster and more accurate than `eig`, and it
returns eigenvalues in ascending order with orthonormal eigenvectors, so
rebuilding is just `V @ diag(w_clipped) @ V.T`. Clipping to zero gives the
nearest PSD matrix in Frobenius norm (the Higham projection).

```python
import numpy as np

w, V = np.linalg.eigh(C)

w_psd = np.clip(w, 0.0, None)
C_psd = V @ np.diag(w_psd) @ V.T
C_psd = (C_psd + C_psd.T) / 2

keep = np.zeros_like(w_psd)
keep[-10:] = w_psd[-10:]
rank10 = V @ np.diag(keep) @ V.T
rank10 = (rank10 + rank10.T) / 2

answers = {
    "n_negative": int((w < 0).sum()),
    "C_psd": C_psd,
    "rank10": rank10,
    "rel_error_rank10": float(np.linalg.norm(C_psd - rank10) /
                              np.linalg.norm(C_psd)),
}
```